# Math Reasoning LLM - Colab Launcher

This notebook serves as the operational launcher for training, evaluation, and data generation on Google Colab (G4 / A100 / L4 runtimes).

> **Architecture Rules:**
> 1. **No Training Logic in Notebook Cells:** All post-training, data synthesis, and evaluation runs inside modular scripts in `scripts/`.
> 2. **Persistence on Google Drive:** Artifacts (`checkpoints/`, `outputs/`, `logs/`, `data/`) are saved directly to `My Drive/math-reasoning-llm/`.
> 3. **High-Speed Model Caching:** Hugging Face model cache remains on Colab's local SSD disk (`/content/hf_cache`), NEVER on Google Drive.
> 4. **Zero-Secret Commits:** Hugging Face token is loaded directly from Google Colab Secrets (`HF_TOKEN`).

## Step 1: Mount Google Drive
Persists checkpoints, generated reasoning datasets, and run outputs across Colab disconnects.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/math-reasoning-llm')
for subfolder in ['checkpoints', 'outputs', 'logs', 'data']:
    (DRIVE_PROJECT_ROOT / subfolder).mkdir(parents=True, exist_ok=True)

print(f"Google Drive mounted successfully at: {DRIVE_PROJECT_ROOT}")

## Step 2: Clone or Pull Repository
Clones fresh code from GitHub directly into `/content/math-reasoning-llm`.

In [ ]:
import os

REPO_URL = "https://github.com/NashLucas/math-reasoning-llm.git"
REPO_DIR = "/content/math-reasoning-llm"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists. Pulling latest commits...")
    %cd {REPO_DIR}
    !git checkout main
    !git pull origin main

%cd {REPO_DIR}
!git status

## Step 3: Hardware Diagnostics & Environment Setup
Inspect the allocated GPU and configure local Hugging Face caching and environment variables.

In [ ]:
import os
import torch
from google.colab import userdata

# 1. Read Hugging Face token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print("Hugging Face token loaded from Colab Secrets.")
except Exception as e:
    print("WARNING: HF_TOKEN not found in Colab Secrets. Add it under the Key icon on the left panel.")

# 2. Configure Local HF Cache on Colab SSD
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['APP_PROFILE'] = 'colab'
os.makedirs('/content/hf_cache', exist_ok=True)

# 3. Hardware Diagnostics
print("=" * 60)
print("HARDWARE DIAGNOSTICS")
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Detected: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU (G4/A100).")
print("=" * 60)

## Step 4: Install Dependencies
Installs project packages on top of Colab's pre-installed PyTorch.

In [ ]:
# Install core requirements
!pip install -r requirements.txt

# Optional: test vLLM installation
# !pip install -r requirements-vllm.txt

## Step 5: Verify Configurations & Stage Paths
Runs `src/config.py` to confirm that all pipeline stages resolve to Google Drive under the `colab` profile.

In [ ]:
!python src/config.py

## Step 6: Script Execution Console
Execute any pipeline script below. For long-running jobs (Phase 4 SFT, Phase 6 GRPO), you can also use Colab Pro's built-in terminal.

In [ ]:
# Example: Run Day 3 dummy verification to test run provenance
!python scripts/dummy_run.py

# Future stage commands:
# !python scripts/prepare_data.py
# !python scripts/evaluate.py --stage baseline --preset greedy
# !python scripts/generate_reasoning_data.py
# !python scripts/train_sft.py
# !python scripts/train_grpo.py